# DEEPX Tutorial 10 — PP-OCRv6 on DEEPX NPU

This tutorial introduces PP-OCRv6 and demonstrates the complete ONNX-to-DXNN OCR workflow on a DEEPX NPU. PP-OCRv6_tiny is the selected target tier because it keeps compilation and classroom turnaround short.

The workflow is divided into stages that can be checked independently:

1. download the original ONNX models;
2. replace dynamic input dimensions with fixed shapes;
3. verify that the fixed models preserve the ONNX results;
4. compile the models to DXNN;
5. run detection and text recognition as one application.

> The original <code>paddleocr.ipynb</code> is kept unchanged. This reviewed notebook uses separate configuration and output directories.

## Learning objectives

After completing this tutorial, you will be able to:

- explain the PP-OCRv6 architecture and choose between tiny, small, and medium tiers;
- explain the detection → recognition OCR pipeline;
- explain why one dynamic recognition model becomes six fixed-shape DXNN models;
- safely download and validate the source ONNX files;
- create calibration configurations whose preprocessing matches the application;
- compile models with <code>dxcom</code> without hiding failures;
- inspect the generated DXNN artifacts;
- run the reviewed camera application and identify practical OCR accuracy limits.

## 1. PP-OCRv6 Overview

### 1.1. What is OCR?

**Optical Character Recognition** (OCR) is the technology that converts different types of documents (scanned paper documents, PDF files, or images captured by a digital camera) into editable and searchable data.

Think of it as giving "eyes" to your AI. It generally works in a two-step pipeline:
1. Text Detection: Locating where the text is in an image (drawing a box around it).
2. Text Recognition: Deciphering what the characters inside that box are.

<img src="https://miro.medium.com/v2/resize:fit:1400/1*2hxwOTzkZQh6EDJDPj4_xg.png" style="max-width: 1000px;">

### 1.2. What is PP-OCRv6?

PaddleOCR is an ultra-lightweight, open-source OCR system developed by Baidu based on the PaddlePaddle framework.

[PP-OCRv6](https://github.com/PaddlePaddle/PaddleOCR) is the latest generation of PaddleOCR's universal OCR model family. It uses the new **PPLCNetV4** backbone for both text detection and text recognition and provides three deployment tiers from edge devices to servers.

Key improvements described in the [official PP-OCRv6 technical documentation](https://www.paddleocr.ai/latest/en/version3.x/algorithm/PP-OCRv6/PP-OCRv6.html) include:

- **one scalable model family:** tiny, small, and medium tiers range from 1.5M to 34.5M parameters;
- **unified multilingual recognition:** small and medium support 50 languages in one model, while tiny supports 49 languages and excludes Japanese;
- **improved text detection:** RepLKFPN expands the receptive field while reducing neck parameters;
- **improved text recognition:** EncoderWithLightSVTR combines local context and global attention, while CTC provides efficient parallel decoding; and
- **specialized-scene coverage:** the official evaluation includes handwriting, rotated and artistic text, digital displays, dot-matrix characters, tire prints, and other industrial text.

<img src="assets/ppocrv6-backbone.jpg" style="max-width: 1000px; width: 100%;" alt="PPLCNetV4 backbone design for PP-OCRv6 recognition and detection">

*PPLCNetV4 uses task-adaptive downsampling: recognition preserves horizontal sequence information, while detection produces multi-scale feature maps. Source: [official PaddleOCR PP-OCRv6 documentation](https://github.com/PaddlePaddle/PaddleOCR/blob/main/docs/version3.x/algorithm/PP-OCRv6/PP-OCRv6.md).*

### 1.3. Tiny, small, and medium

The tiers use the same PP-OCRv6 design at different scales. A larger tier generally improves difficult-case accuracy but increases model size and compute cost.

| Tier | Parameters | Intended target | Detection Hmean (%) | Recognition accuracy (%) | Intel Xeon OpenVINO (s/image) | NVIDIA A100 PaddlePaddle (s/image) |
|---|---:|---|---:|---:|---:|---:|
| **tiny** | 1.5M | Edge / IoT | 80.6 | 73.5 | **0.20** | **0.13** |
| **small** | 7.7M | Mobile / desktop | 84.1 | 81.3 | 0.59 | 0.25 |
| **medium** | 34.5M | Server / highest accuracy | **86.2** | **83.2** | 1.40 | 0.29 |

> **How to read the table:** accuracy values come from PaddleOCR's internal multi-scenario benchmark. Speed is end-to-end seconds per image over 200 general and document images and includes image I/O, preprocessing, post-processing, and inference. These CPU/GPU numbers explain the relative tier trade-off; they are **not DEEPX NPU measurements** and do not predict DX-COM compile time.

<img src="assets/ppocrv6-performance.png" style="max-width: 1100px; width: 100%;" alt="Official PP-OCRv6 detection and recognition accuracy comparison">

*Left: average text-detection Hmean. Right: weighted-average text-recognition accuracy. Source: [official PaddleOCR PP-OCRv6 documentation](https://github.com/PaddlePaddle/PaddleOCR/blob/main/docs/version3.x/algorithm/PP-OCRv6/PP-OCRv6.md).*

### 1.4. Why this tutorial selects PP-OCRv6_tiny

This tutorial selects the **tiny** tier to minimize model preparation and compilation time during a class. Its 1.5M-parameter footprint also makes it the natural starting point for edge and IoT deployment. This is a tutorial-time decision, not a claim that tiny is the best production model. Choose small or medium when the measured accuracy gain justifies the additional resources.

Tiny makes two important trade-offs:

1. it has lower official detection and recognition accuracy than small and medium; and
2. it supports 49 languages and excludes Japanese, whereas small and medium support 50.

> **Implementation status:** Section 4 downloads the official PP-OCRv6_tiny detector and recognizer ONNX models. Before treating the complete camera application as PP-OCRv6, also validate its recognition dictionary, preprocessing, and post-processing against these model contracts.

<img src="assets/ppocrv6-detection-comparison.jpg" style="max-width: 1000px; width: 100%;" alt="Official PP-OCRv6 medium text detection comparison on industrial and difficult text">

*This official qualitative figure uses PP-OCRv6_medium and illustrates difficult detection scenarios; it is not a tiny-tier accuracy result. Source: [official PaddleOCR PP-OCRv6 documentation](https://github.com/PaddlePaddle/PaddleOCR/blob/main/docs/version3.x/algorithm/PP-OCRv6/PP-OCRv6.md).*

## 2. Prepare the tutorial environment

This notebook reads <code>config.json</code> through the shared tutorial path helper. It does not assume that the SDK is installed inside the tutorial repository.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

root_path = os.environ.get("ROOT_PATH")
if not root_path:
    raise EnvironmentError(
        "ROOT_PATH is not set. Start JupyterLab with ./run-jupyter-lab.sh."
    )

TUTORIAL_ROOT = Path(root_path).expanduser().resolve()
sys.path.insert(0, str(TUTORIAL_ROOT))

from tutorial_paths import load_tutorial_path_vars, print_tutorial_paths

globals().update(load_tutorial_path_vars(TUTORIAL_ROOT))
print_tutorial_paths(TUTORIAL_ROOT)

TUTORIAL_DIR = TUTORIAL_ROOT / "notebooks" / "T10-demo-paddleocr"
MODEL_DIR = TUTORIAL_DIR / "models"
CONFIG_DIR = TUTORIAL_DIR / "configs_v6"
OUTPUT_DIR = TUTORIAL_DIR / "outputs" / "paddleocr_v6"

DX_COMPILER_VENV = DX_COMPILER_DIR / "venv-dx-compiler-local"
DXCOM_PATH = DX_COMPILER_VENV / "bin" / "dxcom"
DXPARSE_PATH = Path("/usr/local/bin/dxparse")
DX_ENGINE_PACKAGE_DIR = DX_RT_DIR / "python_package"

for directory in (MODEL_DIR, CONFIG_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

for required in (TUTORIAL_DIR, DXCOM_PATH, DX_ENGINE_PACKAGE_DIR):
    if not required.exists():
        raise FileNotFoundError(f"Required path was not found: {required}")

print(f"Tutorial directory : {TUTORIAL_DIR}")
print(f"Model directory    : {MODEL_DIR}")
print(f"Config directory   : {CONFIG_DIR}")
print(f"Output directory   : {OUTPUT_DIR}")

### 2.1. Install Python packages into the current uv environment

The Jupyter environment was created with uv and may not contain the <code>pip</code> module. The next cell therefore uses <code>uv pip install --python ...</code> instead of <code>%pip</code>.

Re-running the cell is safe: uv reuses packages that already satisfy the requirements.

In [ ]:
UV_PATH = shutil.which("uv")
if UV_PATH is None:
    raise FileNotFoundError(
        "uv is not installed. Install uv, then restart JupyterLab with ./run-jupyter-lab.sh."
    )

NOTEBOOK_PACKAGES = [
    "onnx",
    "onnxruntime",
    "onnxsim",
    "opencv-python",
    "pillow",
    "pyclipper",
    "shapely",
    "torch",
]

install_command = [
    UV_PATH, "pip", "install",
    "--python", sys.executable,
    *NOTEBOOK_PACKAGES,
]
print("Equivalent terminal command:")
print(" ".join(map(str, install_command)))
subprocess.run(install_command, check=True)

## 3. Understand the OCR pipeline

The currently validated PP-OCRv6 reference implementation in this notebook uses two model stages. The same high-level detection and recognition flow also applies to the PP-OCRv6 migration target.

| Stage | Input | Output | Purpose |
|---|---|---|---|
| Text detection | Full image, 640 × 640 | Text polygons | Find text regions |
| Text recognition | One text crop, height 48 | Character sequence | Convert pixels to text |

<img src="assets/ocr-workflow.jpg" style="max-width: 980px;" alt="PP-OCR workflow">

To apply PaddleOCR to DX NPU, following 4 steps are required:

1. Download PaddleOCR ONNX models

2. Fix the dynamic input shape

3. Compile ONNX to *.dxnn for DX NPU

4. Implement OCR application with DEEPX-SDK

## 4. Step #1 - Download and inspect the resources

### 4.1. Download the ONNX models

This tutorial downloads the **tiny** detector and recognizer directly from the official PaddlePaddle ONNX repositories on Hugging Face:

- [PP-OCRv6_tiny_det_onnx](https://huggingface.co/PaddlePaddle/PP-OCRv6_tiny_det_onnx)
- [PP-OCRv6_tiny_rec_onnx](https://huggingface.co/PaddlePaddle/PP-OCRv6_tiny_rec_onnx)

Only the **DET** and **REC** ONNX models are downloaded. The optional text-line orientation classifier is not used in this tutorial.

For repeatable tutorial results, each URL is pinned to a specific official repository revision. Downloads use HTTPS certificate verification, write to a temporary <code>.part</code> file, and verify the SHA-256 checksum before replacing the destination. A matching existing file is reused; a stale or different file is downloaded again.

In [ ]:
import hashlib
from urllib.request import Request, urlopen

MODEL_SOURCES = {
    "det.onnx": {
        "repository": "PaddlePaddle/PP-OCRv6_tiny_det_onnx",
        "revision": "2ba1506c0380b8f0b03dd142459aac66d4421f6c",
        "sha256": "193bab7a04fca699a6c82e6abb5b81bdb28177f0abd4062552b04908dafb19f8",
    },
    "rec.onnx": {
        "repository": "PaddlePaddle/PP-OCRv6_tiny_rec_onnx",
        "revision": "2612ab37152ae0a677521bae4e1e3d4fb4cf7c30",
        "sha256": "9ef676d6ed3c88256a2d92c640c44f25b0c40947e111b14b8be8f594091563e6",
    },
}

def sha256sum(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as model_file:
        for chunk in iter(lambda: model_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def download_model(source: dict, destination: Path) -> Path:
    expected_sha256 = source["sha256"]
    if destination.is_file() and sha256sum(destination) == expected_sha256:
        print(f"SKIP: {destination.name} is already the verified PP-OCRv6 tiny model "
              f"({destination.stat().st_size / 1_000_000:.1f} MB)")
        return destination
    if destination.exists():
        print(f"REPLACE: {destination.name} does not match the selected tiny model")

    url = (
        f"https://huggingface.co/{source['repository']}/resolve/"
        f"{source['revision']}/inference.onnx"
    )

    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.unlink(missing_ok=True)

    request = Request(url, headers={"User-Agent": "dx-tutorials"})
    try:
        with urlopen(request, timeout=120) as response, temporary.open("wb") as output:
            expected = response.headers.get("Content-Length")
            shutil.copyfileobj(response, output)
        if temporary.stat().st_size == 0:
            raise RuntimeError(f"Downloaded file is empty: {destination.name}")
        if expected and temporary.stat().st_size != int(expected):
            raise RuntimeError(
                f"Incomplete download for {destination.name}: "
                f"{temporary.stat().st_size} of {expected} bytes"
            )
        actual_sha256 = sha256sum(temporary)
        if actual_sha256 != expected_sha256:
            raise RuntimeError(
                f"Checksum mismatch for {destination.name}: {actual_sha256}"
            )
        temporary.replace(destination)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise

    print(f"DOWNLOADED: {destination.name} "
          f"({destination.stat().st_size / 1_000_000:.1f} MB)")
    return destination

ONNX_MODELS = {
    name: download_model(source, MODEL_DIR / name)
    for name, source in MODEL_SOURCES.items()
}

### 4.2. Download the calibration dataset

Download and extract the OCR calibration images into this tutorial directory. The archive creates the following directories:

```text
models/
├── det_dataset/
└── rec_dataset/
```

The archive file acts as the completion marker. If <code>ocr-dataset.tar.gz</code> already exists, both the download and extraction are skipped.

In [ ]:
CALIBRATION_DATASET_URL = (
    "https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/ocr-dataset.tar.gz"
)
CALIBRATION_ARCHIVE = TUTORIAL_DIR / "ocr-dataset.tar.gz"

if CALIBRATION_ARCHIVE.is_file():
    print(f"SKIP: {CALIBRATION_ARCHIVE.name} already exists")
    print("      Download and extraction are not required.")
else:
    temporary_archive = CALIBRATION_ARCHIVE.with_name(
        CALIBRATION_ARCHIVE.name + ".part"
    )
    temporary_archive.unlink(missing_ok=True)

    try:
        subprocess.run(
            ["wget", "--https-only", "-O", str(temporary_archive),
             CALIBRATION_DATASET_URL],
            cwd=TUTORIAL_DIR,
            check=True,
        )
        subprocess.run(
            ["tar", "-xzf", str(temporary_archive), "-C", str(TUTORIAL_DIR)],
            check=True,
        )
        temporary_archive.replace(CALIBRATION_ARCHIVE)
    except Exception:
        temporary_archive.unlink(missing_ok=True)
        raise

    print(f"DOWNLOADED: {CALIBRATION_ARCHIVE}")
    print(f"EXTRACTED : {MODEL_DIR / 'det_dataset'}")
    print(f"            {MODEL_DIR / 'rec_dataset'}")

for dataset_dir in (MODEL_DIR / "det_dataset", MODEL_DIR / "rec_dataset"):
    if not dataset_dir.is_dir():
        raise FileNotFoundError(
            f"Expected dataset directory was not found: {dataset_dir}. "
            f"Remove {CALIBRATION_ARCHIVE} and run this cell again."
        )

In [ ]:
import onnx

def tensor_shape(value_info):
    return [
        dim.dim_value if dim.HasField("dim_value") else dim.dim_param or "dynamic"
        for dim in value_info.type.tensor_type.shape.dim
    ]

for name, model_path in ONNX_MODELS.items():
    onnx.checker.check_model(str(model_path))
    model = onnx.load(model_path, load_external_data=False)
    inputs = [(value.name, tensor_shape(value)) for value in model.graph.input]
    outputs = [(value.name, tensor_shape(value)) for value in model.graph.output]
    print(f"{name:10} input={inputs} output={outputs} nodes={len(model.graph.node)}")

The batch size and image dimensions are dynamic in the source models.

Dynamic shapes are useful for a general ONNX Runtime application, but each DEEPX compilation requires a concrete input shape.

## 5. Step #2 - Fix the Dynamic Input Shape

The following table is the single source of truth for the fixed models. Shapes use NCHW order: batch, channels, height, width.

In [ ]:
FIXED_MODEL_SPECS = [
    {"name": "det_fixed",            "source": "det.onnx", "shape": [1, 3, 640, 640]},
    {"name": "rec_fixed_ratio_2_5",  "source": "rec.onnx", "shape": [1, 3, 48, 120]},
    {"name": "rec_fixed_ratio_5",    "source": "rec.onnx", "shape": [1, 3, 48, 240]},
    {"name": "rec_fixed_ratio_10",   "source": "rec.onnx", "shape": [1, 3, 48, 480]},
    {"name": "rec_fixed_ratio_15",   "source": "rec.onnx", "shape": [1, 3, 48, 720]},
    {"name": "rec_fixed_ratio_25",   "source": "rec.onnx", "shape": [1, 3, 48, 1200]}
]

for spec in FIXED_MODEL_SPECS:
    print(f"{spec['name']:22} {spec['shape']}")

### 5.1. Fix input shape of TEXT Recognition Model

**Why do we need 5 different models?**

Since NPU requires a fixed input shape, treating a short word (like 'Hi') and a long sentence in the same 1280-width box would result in excessive padding and loss of detail.

By creating 'buckets' of different aspect ratios, we can process text more efficiently and accurately. That's why we use six separate models with different aspect ratios to improve recognition accuracy.

For each case, we select and apply the model that best matches the ratio of the detected text.

<img src="assets/ocr-ratio.png" style="max-width: 800px;">

The following gif animation shows which text recognition model is matched based on the ratio of text actually detected.

<img src="assets/ocr-ratio.gif" style="max-width: 800px;">

### 5.2. Fix the Dynamic Input Shape using `onnxsim`

The notebook invokes ONNX Simplifier through the current kernel's Python interpreter. For example, the first conversion is equivalent to:

~~~bash
python -m onnxsim models/det.onnx models/det_fixed.onnx           --overwrite-input-shape x:1,3,640,640
python -m onnxsim models/det.onnx models/rec_fixed_ratio_2_5.onnx --overwrite-input-shape x:1,3,48,120
python -m onnxsim models/det.onnx models/rec_fixed_ratio_5.onnx   --overwrite-input-shape x:1,3,48,240
...
~~~

Existing non-empty fixed models are checked and reused.

In [ ]:
import shlex

def fixed_model_path(spec) -> Path:
    return MODEL_DIR / f"{spec['name']}.onnx"

for spec in FIXED_MODEL_SPECS:
    source = MODEL_DIR / spec["source"]
    destination = fixed_model_path(spec)

    if destination.is_file() and destination.stat().st_size > 0:
        try:
            onnx.checker.check_model(str(destination))
            print(f"SKIP: {destination.name} already exists and is valid")
            continue
        except Exception:
            print(f"REBUILD: {destination.name} is not a valid ONNX model")
            destination.unlink()

    shape_argument = "x:" + ",".join(map(str, spec["shape"]))
    command = [
        sys.executable, "-m", "onnxsim",
        str(source), str(destination),
        "--overwrite-input-shape", shape_argument,
    ]
    print("\n" + shlex.join(command))
    subprocess.run(command, cwd=TUTORIAL_DIR, check=True)
    onnx.checker.check_model(str(destination))

### 5.3. Verify shapes and numerical equivalence

Structural validation alone is not enough. The next cells:

1. check every fixed ONNX model;
2. confirm its exact input shape;
3. compare ONNX Runtime outputs for detection and two representative recognition buckets.

All recognition buckets come from the same source graph. The ratio-2.5 and ratio-5 models are used for the numerical recognition check to keep the tutorial validation quick.

In [ ]:
for spec in FIXED_MODEL_SPECS:
    path = fixed_model_path(spec)
    onnx.checker.check_model(str(path))
    model = onnx.load(path, load_external_data=False)
    actual_shape = tensor_shape(model.graph.input[0])
    if actual_shape != spec["shape"]:
        raise AssertionError(
            f"{path.name}: expected {spec['shape']}, got {actual_shape}"
        )
    print(f"PASS: {path.name:28} input={actual_shape}")

In [ ]:
import numpy as np
import onnxruntime as ort

NUMERICAL_CHECKS = [
    FIXED_MODEL_SPECS[0],  # detection
    FIXED_MODEL_SPECS[1],  # recognition ratio 2.5
    FIXED_MODEL_SPECS[2],  # recognition ratio 5
]

rng = np.random.default_rng(2026)

for spec in NUMERICAL_CHECKS:
    source_path = MODEL_DIR / spec["source"]
    fixed_path = fixed_model_path(spec)
    sample = rng.random(spec["shape"], dtype=np.float32)

    source_session = ort.InferenceSession(
        str(source_path), providers=["CPUExecutionProvider"]
    )
    fixed_session = ort.InferenceSession(
        str(fixed_path), providers=["CPUExecutionProvider"]
    )
    source_output = source_session.run(None, {"x": sample})
    fixed_output = fixed_session.run(None, {"x": sample})

    if len(source_output) != len(fixed_output):
        raise AssertionError(f"Output count changed for {spec['name']}")

    max_error = max(
        float(np.max(np.abs(reference - candidate)))
        for reference, candidate in zip(source_output, fixed_output)
    )
    equivalent = all(
        np.allclose(reference, candidate, rtol=1e-4, atol=1e-5)
        for reference, candidate in zip(source_output, fixed_output)
    )
    print(f"{spec['name']:22} equivalent={equivalent} max_abs_error={max_error:.3e}")
    if not equivalent:
        raise AssertionError(f"Numerical output changed for {spec['name']}")

## 6. Step #3 - Compile ONNX to *.dxnn for DX NPU

### 6.1. Create DX-COM calibration configurations

Calibration preprocessing must match application preprocessing. A mismatch in color order, scaling, normalization, or layout can reduce accuracy even when compilation succeeds.

| Model | Calibration images | Resize | Mean / standard deviation |
|---|---|---:|---|
| Detection | <code>det_dataset</code> | 640 × 640 | ImageNet values |
| Recognition | Matching ratio bucket | Fixed width × 48 | [0.5, 0.5, 0.5] |


In [ ]:
import json

DET_DATASET = TUTORIAL_DIR / "models" / "det_dataset"
REC_DATASET = TUTORIAL_DIR / "models" / "rec_dataset"

required_datasets = [
    DET_DATASET,
    REC_DATASET / "ratio_5",
    REC_DATASET / "ratio_15",
    REC_DATASET / "ratio_25",
]
for dataset in required_datasets:
    if not dataset.is_dir():
        raise FileNotFoundError(f"Calibration dataset was not found: {dataset}")

def image_count(directory: Path) -> int:
    extensions = {".jpeg", ".jpg", ".png"}
    return sum(
        path.is_file() and path.suffix.lower() in extensions
        for path in directory.iterdir()
    )

for dataset in required_datasets:
    count = image_count(dataset)
    if count == 0:
        raise RuntimeError(f"No calibration images were found in {dataset}")
    print(f"{dataset.relative_to(TUTORIAL_DIR)!s:28} {count:4} images")

In [ ]:
def preprocessing(width, height, mean, std):
    return [
        {"resize": {"width": width, "height": height}},
        {"convertColor": {"form": "BGR2RGB"}},
        {"div": {"x": 255}},
        {"normalize": {"mean": mean, "std": std}},
        {"transpose": {"axis": [2, 0, 1]}},
        {"expandDim": {"axis": 0}},
    ]

def calibration_config(shape, dataset, calibration_num, mean, std):
    _, _, height, width = shape
    return {
        "inputs": {"x": shape},
        "calibration_num": calibration_num,
        "calibration_method": "ema",
        "default_loader": {
            "dataset_path": str(dataset),
            "file_extensions": ["jpeg", "jpg", "png", "JPEG"],
            "preprocessings": preprocessing(width, height, mean, std),
        }
    }

REC_DATASET_BY_BUCKET = {
    3: REC_DATASET / "ratio_5",
    5: REC_DATASET / "ratio_5",
    10: REC_DATASET / "ratio_15",
    15: REC_DATASET / "ratio_15",
    25: REC_DATASET / "ratio_25",
    35: REC_DATASET / "ratio_25",
}

COMPILE_SPECS = []

for spec in FIXED_MODEL_SPECS:
    if spec["name"] == "det_fixed":
        config = calibration_config(
            spec["shape"], DET_DATASET, 100,
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225],
        )
    else:
        bucket = int(spec["name"].rsplit("_", 1)[1])
        config = calibration_config(
            spec["shape"], REC_DATASET_BY_BUCKET[bucket], 80,
            [0.5, 0.5, 0.5],
            [0.5, 0.5, 0.5],
        )

    config_path = CONFIG_DIR / f"{spec['name']}.json"
    config_path.write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
    COMPILE_SPECS.append({
        **spec,
        "onnx": fixed_model_path(spec),
        "config": config_path,
        "output_dir": OUTPUT_DIR / spec["name"],
    })
    print(f"WROTE: {config_path.relative_to(TUTORIAL_DIR)}")

Inspect at least one configuration before compiling. In particular, check NCHW input shape, calibration dataset, resize size, channel order, normalization, and transpose order.

In [ ]:
example_config = COMPILE_SPECS[0]["config"]
print(example_config.read_text(encoding="utf-8"))

### 6.2. Compile ONNX models to DXNN

Each model is written to its own directory. A valid existing DXNN file is skipped so that a repeated class does not spend time recompiling it.

The Python call is only used to display and execute the command reliably. For example, a compile is equivalent to:

~~~bash
source ~/dx-all-suite/dx-compiler/venv-dx-compiler-local/bin/activate
dxcom -m <fixed-model.onnx> \
      -c <calibration-config.json> \
      -o <output-directory> \
      --gen_log \
      --export_html
~~~

Unlike the original notebook, compiler output and failures are not redirected or converted into a successful cell.

In [ ]:
# Keep all entries for a complete OCR application.
# During a short class, you may select a smaller list only to demonstrate compilation.
SELECTED_COMPILES = [spec["name"] for spec in COMPILE_SPECS]
print("Models selected for compilation:")
for name in SELECTED_COMPILES:
    print(" -", name)

In [ ]:
def expected_dxnn(spec) -> Path:
    return spec["output_dir"] / f"{spec['name']}.dxnn"

def compile_model(spec) -> Path:
    artifact = expected_dxnn(spec)
    if artifact.is_file() and artifact.stat().st_size > 0:
        print(f"SKIP: {artifact.relative_to(TUTORIAL_DIR)} already exists")
        return artifact

    spec["output_dir"].mkdir(parents=True, exist_ok=True)
    command = [
        str(DXCOM_PATH),
        "-m", str(spec["onnx"]),
        "-c", str(spec["config"]),
        "-o", str(spec["output_dir"]),
        "--gen_log",
        "--export_html",
    ]
    print("\nEquivalent terminal command:")
    print(f"source {shlex.quote(str(DX_COMPILER_VENV / 'bin' / 'activate'))}")
    print(shlex.join(command))
    subprocess.run(command, cwd=TUTORIAL_DIR, check=True)

    if not artifact.is_file() or artifact.stat().st_size == 0:
        raise FileNotFoundError(f"DX-COM did not create {artifact}")
    return artifact

COMPILED_MODELS = {}
for spec in COMPILE_SPECS:
    if spec["name"] in SELECTED_COMPILES:
        COMPILED_MODELS[spec["name"]] = compile_model(spec)

### 6.3. Verify compiled artifacts

A complete OCR application needs exactly six DXNN files: detection and five recognition buckets. The cell fails early if any required file is missing.

In [ ]:
REQUIRED_DXNN_NAMES = [spec["name"] for spec in COMPILE_SPECS]
missing = []

for spec in COMPILE_SPECS:
    artifact = expected_dxnn(spec)
    if artifact.is_file() and artifact.stat().st_size > 0:
        print(f"PASS: {artifact.relative_to(TUTORIAL_DIR)!s:55} "
              f"{artifact.stat().st_size / 1_000_000:7.2f} MB")
    else:
        missing.append(artifact)

if missing:
    raise FileNotFoundError(
        "Missing DXNN files:\n" + "\n".join(f" - {path}" for path in missing)
    )

Use <code>dxparse</code> to inspect the two model roles. Recognition ratio 2.5 is representative; the other recognition files differ mainly in fixed input width.

In [ ]:
if not DXPARSE_PATH.is_file():
    raise FileNotFoundError(f"dxparse was not found: {DXPARSE_PATH}")

for model_name in ("det_fixed", "rec_fixed_ratio_2_5"):
    model_path = next(
        expected_dxnn(spec) for spec in COMPILE_SPECS if spec["name"] == model_name
    )
    command = [str(DXPARSE_PATH), "-m", str(model_path), "-v"]
    print("\n" + "=" * 72)
    print(shlex.join(command))
    subprocess.run(command, cwd=TUTORIAL_DIR, check=True)

## 7. Step #4 - Test the OCR application with DEEPX-SDK

The reusable Python application is located in <code>app/</code>. It uses the compiled PP-OCRv6 tiny models from <code>outputs/paddleocr_v6/</code> and processes each frame as follows:

<img src="assets/ppocrv6-camera-pipeline.png" alt="PP-OCRv6 camera inference pipeline" style="max-width: 1000px; width: 100%; height: auto;">

The detector and five recognition engines are loaded once and remain resident.

### 7.1 Install the application dependencies

Use the same uv-managed Python environment as the Notebook. The first command installs the normal Python dependencies from <code>app/requirements.txt</code>; the second installs the local DX-RT Python package that provides <code>dx_engine</code>. Re-running these commands is safe.

In [ ]:
APP_DIR = TUTORIAL_DIR / "app"
APP_REQUIREMENTS = APP_DIR / "requirements.txt"
CAMERA_APP = APP_DIR / "camera_app.py"
DXRT_PYTHON_PACKAGE = DX_ENGINE_PACKAGE_DIR

for required in (APP_DIR, APP_REQUIREMENTS, CAMERA_APP, DXRT_PYTHON_PACKAGE):
    if not required.exists():
        raise FileNotFoundError(f"Required application path was not found: {required}")

install_commands = [
    [UV_PATH, "pip", "install", "--python", sys.executable, "-r", str(APP_REQUIREMENTS)],
    [UV_PATH, "pip", "install", "--python", sys.executable, str(DXRT_PYTHON_PACKAGE)],
]

for command in install_commands:
    print("Equivalent terminal command:")
    print(shlex.join(command))
    subprocess.run(command, check=True)

import cv2
import dx_engine

print("OpenCV   :", cv2.__version__)
print("dx_engine:", dx_engine.__file__)

### 7.2 Application structure and safeguards

The Notebook now uses the maintained files under <code>app/</code> instead of generating another runner.

| File | Responsibility |
|---|---|
| <code>app/ocr_engine.py</code> | DXNN loading, DET post-processing, crop extraction, five-bucket REC routing, and CTC decoding |
| <code>app/camera_app.py</code> | CLI validation, 640×480 camera capture, OpenCV preview, BBOX and multilingual text overlay |
| <code>app/run_camera.sh</code> | Starts the application with the repository's uv environment |
| <code>app/assets/ppocrv6_tiny_dict.txt</code> | Official tiny dictionary matching the 6906-class REC output |

The application fails clearly for missing models, invalid tensor shapes, a mismatched dictionary, or a camera-open/read failure. Existing DXNN files are never modified.

### 7.3 Run a camera-free application smoke test

The <code>--check</code> mode does not open a camera or create a GUI window. It loads the packaged application and runs one synthetic inference through the detector and all five recognizers. This verifies the NPU runtime, input/output tensor contracts, and tiny dictionary class count before the live demo.

The command executed below is equivalent to running <code>python app/camera_app.py --check</code> with explicit model and dictionary paths.

In [ ]:
APP_DICTIONARY = APP_DIR / "assets" / "ppocrv6_tiny_dict.txt"
REQUIRED_MODEL_NAMES = [
    "det_fixed",
    "rec_fixed_ratio_2_5",
    "rec_fixed_ratio_5",
    "rec_fixed_ratio_10",
    "rec_fixed_ratio_15",
    "rec_fixed_ratio_25",
]

required_files = [CAMERA_APP, APP_DICTIONARY]
required_files.extend(
    OUTPUT_DIR / name / f"{name}.dxnn"
    for name in REQUIRED_MODEL_NAMES
)
missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Application test files are missing:\n"
        + "\n".join(f" - {path}" for path in missing_files)
    )

smoke_test_command = [
    sys.executable,
    str(CAMERA_APP),
    "--check",
    "--model-dir", str(OUTPUT_DIR),
    "--dictionary", str(APP_DICTIONARY),
]
print("Equivalent terminal command:")
print(shlex.join(smoke_test_command))
subprocess.run(smoke_test_command, cwd=TUTORIAL_DIR, check=True)

### 7.4 Test the 640×480 camera application

The live application opens an interactive OpenCV window. Run it in a **JupyterLab Terminal** so that the Notebook kernel is not blocked and the camera can be released cleanly.

The default input is <code>/dev/video0</code> at 640×480 and 15 FPS. Press **q** or **Esc** in the preview window to exit. Use <code>--camera /dev/videoN</code> to select another device.

In [ ]:
camera_command = [
    str(APP_DIR / "run_camera.sh"),
    "--camera", "/dev/video0",
    "--width", "640",
    "--height", "480",
    "--fps", "15",
]

print("Run in a separate JupyterLab Terminal:")
print(f"cd {shlex.quote(str(APP_DIR))}")
print(shlex.join(["./run_camera.sh", *camera_command[1:]]))

<img src="assets/sc-ocr-app.png" style="max-width: 800px;" alt="Expected PP-OCR result">

## 8. Accuracy and performance checklist

A successful compile does not guarantee acceptable OCR accuracy. Validate the whole pipeline with representative images.

| Check | Why it matters | Practical action |
|---|---|---|
| Detection resolution | Small text can disappear at 640 × 640 | Compare recall at the expected camera distance |
| Calibration coverage | Quantization follows calibration statistics | Include real lighting, fonts, blur, and backgrounds |
| Preprocessing parity | Color or normalization mismatch shifts model inputs | Keep compile and runtime preprocessing identical |
| Recognition bucket | Excessive resize or padding hurts characters | Measure text aspect-ratio distribution |
| Recognition confidence | A low threshold shows incorrect text; a high threshold drops text | Tune on a labeled validation set |
| Perspective and orientation | Camera documents are not always flat or upright | Add document orientation and unwarping when needed |
| End-to-end latency | One frame may contain many recognition crops | Report latency against text count, not only FPS |

Do not tune against a single attractive camera sample. Keep a fixed validation set and record detection recall, recognition accuracy, and end-to-end latency for every configuration change.

## 9. Troubleshooting

| Symptom | Check |
|---|---|
| <code>ROOT_PATH is not set</code> | Start JupyterLab with <code>./run-jupyter-lab.sh</code> |
| <code>dxcom</code> path is missing | Complete the DX-COM installation and check <code>config.json</code> |
| Downloaded file is invalid | Delete only the named model and rerun Section 4 |
| Fixed model shape is wrong | Delete that fixed ONNX file and rerun Section 5 |
| Compilation fails | Read the visible DX-COM output and the model output directory's log |
| <code>dx_engine</code> import fails | Rerun Section 7.1 with the current Notebook kernel |
| <code>--check</code> reports a missing model | Compile all six models listed in Section 7.3 |
| Camera cannot open | Check permissions and the selected <code>/dev/video*</code> device |
| OpenCV display error | Run from a graphical JupyterLab Terminal with a valid <code>DISPLAY</code> |
| No text is shown | Check lighting/focus and tune the DET, box, and REC thresholds on representative images |

## 10. Summary

### 10.1. Workflow completed

```text
Official PP-OCRv6 tiny DET + REC ONNX
                    │
                    ▼
       Fix dynamic input dimensions
          ┌─────────┴─────────┐
          │                   │
   DET 640×640       REC width buckets ×5
          └─────────┬─────────┘
                    ▼
     Calibration configuration + DX-COM
                    │
                    ▼
           Six verified DXNN models
                    │
          ┌─────────┴─────────┐
          ▼                   ▼
 camera_app.py --check    640×480 camera
                              │
                              ▼
                   BBOX + recognized text
```

### 10.2. Artifact and verification dashboard

| Stage | Artifact or command | What it verifies |
|---|---|---|
| Source models | <code>det.onnx</code>, <code>rec.onnx</code> | Official revision and SHA-256 checksum |
| Fixed ONNX | One DET and five REC models | Static input shapes and ONNX validity |
| Numerical check | ONNX Runtime comparison | Fixed-shape conversion preserves representative outputs |
| Calibration | <code>configs_v6/*.json</code> | Dataset, resize, color order, normalization, and layout |
| Compilation | <code>dxcom</code> | ONNX-to-DXNN conversion with visible logs and reports |
| Runtime contract | <code>camera_app.py --check</code> | DET and all five REC models execute with the expected tensor contracts |
| Live application | <code>app/run_camera.sh</code> | 640×480 capture, ratio routing, CTC decoding, BBOX, and text overlay |

### 10.3. Runtime model map

| Role | Fixed input | Selection rule |
|---|---:|---|
| DET | `[1, 640, 640, 3]` | Once per camera frame |
| REC 2.5 | `[1, 48, 120, 3]` | Crop aspect ratio ≤ 2.5 |
| REC 5 | `[1, 48, 240, 3]` | Crop aspect ratio ≤ 5 |
| REC 10 | `[1, 48, 480, 3]` | Crop aspect ratio ≤ 10 |
| REC 15 | `[1, 48, 720, 3]` | Crop aspect ratio ≤ 15 |
| REC 25 | `[1, 48, 1200, 3]` | Crop aspect ratio ≤ 25, or fallback for wider crops |

### 10.4. Completion checklist

- [x] Downloaded the official PP-OCRv6 tiny DET and REC ONNX models
- [x] Verified source-model checksums and ONNX structure
- [x] Created one fixed DET model and five fixed REC models
- [x] Compared representative source and fixed ONNX outputs
- [x] Created calibration configurations and compiled the DXNN models
- [x] Inspected the compiled artifacts and tested all six models on the NPU
- [x] Prepared the DET-to-REC OpenCV camera application
- [ ] Measure detection and recognition accuracy on a labeled validation set
- [ ] Tune thresholds with representative camera images
- [ ] Record end-to-end latency, throughput, and host CPU usage

### 10.5. Trying the small or medium tier

To repeat this workflow with a larger PP-OCRv6 tier, download both the detector and recognizer from the matching official PaddlePaddle ONNX repositories:

| Tier | Detection ONNX repository | Recognition ONNX repository |
|---|---|---|
| **small** | [PP-OCRv6_small_det_onnx](https://huggingface.co/PaddlePaddle/PP-OCRv6_small_det_onnx) | [PP-OCRv6_small_rec_onnx](https://huggingface.co/PaddlePaddle/PP-OCRv6_small_rec_onnx) |
| **medium** | [PP-OCRv6_medium_det_onnx](https://huggingface.co/PaddlePaddle/PP-OCRv6_medium_det_onnx) | [PP-OCRv6_medium_rec_onnx](https://huggingface.co/PaddlePaddle/PP-OCRv6_medium_rec_onnx) |


### 10.6. How to improve OCR performance?

This tutorial does not implement pre-processing blocks such as `Document Image Orientation Classification`, `Text Image Unwarping` and `Text Line Orientation Classification`.

If you want higher OCR performance, implement and apply the missing AI models to the OCR AI pipeline.

<img src="assets/ppocrv6-full-pipeline.png" style="max-width: 1200px;">

For higher accuracy:

- Build the calibration dataset with text images from the **actual deployment environment**.
- Cover the real camera, distance, text size, aspect ratio, font, language, lighting, blur, and perspective.
- The current **calibration dataset** has three broad REC buckets: ratio 5, 15, and 25. Split these dataset buckets more finely when deployment text shapes vary widely.
- These calibration buckets are different from the five runtime REC model routes: ratio 2.5, 5, 10, 15, and 25.
- Convert dynamic DET and REC inputs to fixed shapes selected from measured deployment data.
- Keep calibration and runtime preprocessing identical.

| Deployment data | Adjustment |
|---|---|
| Ratios between current buckets | Add intermediate REC buckets |
| Tall, narrow, or long text | Use matching fixed REC shapes |
| Very small text | Evaluate a larger DET input |
| Unusual camera aspect ratio | Match the DET input geometry |
| Distortion after resizing | Adjust shapes and preprocessing |

Before release:

- Compare ONNX and DXNN accuracy on the same labeled dataset.
- Measure detection recall, recognition accuracy, latency, memory, and CPU load.
- Recheck tensor contracts, preprocessing, calibration, and the recognition dictionary.
- Add shapes only when they provide measurable value; more shapes increase build time, storage, and routing complexity.